# Figure 4: Building disease-drug gene/pathway networks

In [131]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from joblib import Parallel, delayed
import math


## 1. Load STRING edges and build weighted graph

In [132]:
G = nx.Graph()
for _, row in string_data.iterrows():
    G.add_edge(row['gene1'], row['gene2'], weight=1-row['weight'])


## 2. Load node types (TB disease genes + DGIdb drug targets)

In [ ]:
node_types = pd.read_csv(
    "/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/figures/figure4/node_types_with_perturbed_genes.tsv",
    sep="\t"
)

node_types['node'] = node_types['node'].astype(str).str.strip()
node_types['type'] = node_types['type'].astype(str).str.strip()

sources = node_types[node_types['type'] == 'source']['node'].tolist()
targets = node_types[node_types['type'] == 'target']['node'].tolist()

sources_in_G = [s for s in sources if s in G]
targets_in_G = [t for t in targets if t in G]


## 3. Remove NaN nodes

In [134]:
nan_nodes = [n for n in G.nodes if isinstance(n, float) and math.isnan(n)]
G.remove_nodes_from(nan_nodes)


## 4. Compute all shortest paths (Dijkstra)

In [135]:
def process_source(src):
    paths = {}
    if src not in G: 
        return paths

    sp = nx.single_source_dijkstra_path(G, src, weight="weight")
    for tgt in targets_in_G:
        if tgt in sp:
            paths[(src, tgt)] = sp[tgt]
    return paths

results = Parallel(n_jobs=-1)(delayed(process_source)(s) for s in sources_in_G)

shortest_paths = {k: v for d in results for k, v in d.items()}


## 5. Compute betweenness centrality on a subgraph of relevant edges

In [136]:
all_edges = [
    (u, v)
    for path in shortest_paths.values()
    for u, v in zip(path[:-1], path[1:])
]

subgraph = G.edge_subgraph(all_edges).copy()
bc_scores = nx.betweenness_centrality(subgraph, weight="weight")


## 6. Select top 2% paths by average BC per source

In [137]:
# Compute avg BC for each path
path_avg_bc = {
    key: np.mean([bc_scores.get(n, 0) for n in path])
    for key, path in shortest_paths.items()
}

# Group by source
paths_by_source = {}
for (src, tgt), score in path_avg_bc.items():
    paths_by_source.setdefault(src, []).append(((src, tgt), score))

# Select top 2%
top_paths_per_source = {}
top_pct = 0.02

for src, lst in paths_by_source.items():
    lst_sorted = sorted(lst, key=lambda x: x[1], reverse=True)
    keep_n = max(1, int(len(lst_sorted)*top_pct))
    top_subset = lst_sorted[:keep_n]
    top_paths_per_source[src] = {pair: shortest_paths[pair] for pair, _ in top_subset}


## 7. Identify intermediate genes (excluding source + target)

In [138]:
intermediate_genes = set()

for paths in top_paths_per_source.values():
    for path in paths.values():
        intermediate_genes.update(path[1:-1])


## 8. Remove hub genes (top 20% degree) without breaking connectivity

#### Important: We remove hub nodes after path extraction, but we do not delete from paths.
#### We simply exclude them from being labeled as "in-path". So, Paths remain connected ↔ hubs remain in the graph but not classified as in-path genes.

In [139]:
deg_cent = nx.degree_centrality(G)
cutoff = sorted(deg_cent.values(), reverse=True)[int(len(deg_cent)*0.20)]
hub_genes = {n for n,v in deg_cent.items() if v >= cutoff}

intermediate_genes = intermediate_genes - hub_genes


## 9. Build Cytoscape edge table (from exact paths)

In [140]:
edge_list = []

for paths in top_paths_per_source.values():
    for path in paths.values():
        for u, v in zip(path[:-1], path[1:]):
            edge_list.append((u, v, "ppi"))


## 10. Add drug–gene edges

In [ ]:
drug_gene_df = pd.read_csv(
    "/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/figures/figure4/drug_gene_table.tsv",
    sep="\t"
)

drug_gene_edges = []
for _, row in drug_gene_df.iterrows():
    gene = str(row["drug_gene"]).strip()
    drug = str(row["drug_name"]).strip()

    if gene in subgraph:
        drug_gene_edges.append((drug, gene, "drug-gene"))

edge_list.extend(drug_gene_edges)


## 11. Build Cytoscape Node Table

In [ ]:
# collect all nodes
all_nodes = set([u for u,_,_ in edge_list] + [v for _,v,_ in edge_list])
# Classification
def classify(node):
    if node in sources: return "source"
    if node in targets: return "target"
    if node in intermediate_genes: return "in-path"
    if node in hub_genes: return "hub"
    if node in drug_gene_df["drug_name"].values: return "drug"
    return "gene"

node_table = pd.DataFrame({
    "shared name": list(all_nodes),
    "type": [classify(n) for n in all_nodes]
})

node_table.to_csv("cytoscape_node_table.csv", index=False)

node_table


,shared name,type
0,mitotane,drug
1,HTR1F,target
2,NPEPL1,target
3,estradiol,drug
4,vemurafenib,drug
...,...,...
358,temsirolimus,drug
359,SPAG4,target
360,raloxifene,drug
361,carbamazepine,drug


## 12. Build Cytoscape Edge table

In [ ]:
edge_rows = []

# 1. Add gene–gene shortest-path edges
for paths in top_paths_per_source.values():
    for path in paths.values():
        for u, v in zip(path[:-1], path[1:]):
            edge_rows.append((u, v, "ppi"))

# 2. Add drug–gene edges
for _, row in drug_gene_df.iterrows():
    gene = str(row["drug_gene"]).strip()
    drug = str(row["drug_name"]).strip()

    # only include drugs whose target gene appears in any path
    if gene in {n for row in edge_rows for n in row[:2]}:
        edge_rows.append((drug, gene, "drug-gene"))

# 3. Construct dataframe
edge_table_df = pd.DataFrame(edge_rows, columns=["Source", "Target", "Interaction"])

# 4. Remove duplicates
edge_table_df = edge_table_df.drop_duplicates()

# 5. Save to file
edge_table_df.to_csv(
    "/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/figures/figure4/cytoscape_edge_table.csv",
    index=False
)

edge_table_df


,Source,Target,Interaction
0,ADRB2,UBC,ppi
1,UBC,RHOA,ppi
3,UBC,PCNA,ppi
5,UBC,CXCR4,ppi
7,UBC,CDKN1A,ppi
...,...,...,...
318,axitinib,PCNA,drug-gene
319,lisuride,SRC,drug-gene
320,methoxsalen,CXCR4,drug-gene
321,lapatinib,CD44,drug-gene
